# ViRead VietOCR V2 persistent Colab worker

After one-time secret setup, run the worker cell when a standard Colab GPU runtime is connected. It polls the ViRead queue, takes only ready Flow 2 jobs, downloads and verifies the exact dataset and parent checkpoint, trains, uploads, benchmarks, and reports status automatically. Do not enter job IDs or edit profile settings. Keep this runtime connected for automatic processing; standard Colab has no supported start-runtime API exposed to this backend. Colab Enterprise is a separate Google Cloud service with its own supported lifecycle API.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import subprocess, sys, os

def secret(name, required=True):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing Colab Secret: {name}")
    return value

API_BASE = secret("VIREAD_API_BASE").rstrip("/")
WORKER_TOKEN = secret("VIREAD_TRAINING_WORKER_TOKEN")
REPOSITORY_REF = secret("VIREAD_REPOSITORY_REF", required=False) or "main"
REPOSITORY_URL = "https://github.com/trancuong3/capstone1.git"
REPO = Path("/content/viread-capstone1")
drive.mount("/content/drive")

if not REPO.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPOSITORY_REF], check=True)
dirty = subprocess.run(["git", "-C", str(REPO), "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
if dirty:
    raise RuntimeError("Worker source checkout is modified; use a clean runtime directory.")
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "ai/vietocr_v2/requirements.txt")], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
from ai.vietocr_v2.colab_worker import run_worker

PERSISTENT_WORK = Path("/content/drive/MyDrive/ViRead/VietOCR-V2-worker")
PERSISTENT_WORK.mkdir(parents=True, exist_ok=True)
print("Worker configured. Starting automatic queue polling; do not print or share secrets.")
run_worker(API_BASE, WORKER_TOKEN, PERSISTENT_WORK, poll_seconds=30)
